# Using Agents in LlamaIndex


In [8]:
from dotenv import load_dotenv
load_dotenv()

from llama_index.llms.openai import OpenAI
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.tools import FunctionTool

FLIGHT_SCHEDULE = {
    ("nairobi", "mombasa"): [
        ("Kenya Airways", "KQ203", "07:15", 85),
        ("Fly540", "5F112", "11:40", 62),
        ("Jambojet", "JM872", "17:05", 58),
    ],
    ("nairobi", "kigali"): [
        ("RwandAir", "WB404", "09:05", 140),
        ("Kenya Airways", "KQ482", "16:20", 155),
    ],
    ("mombasa", "dar es salaam"): [
        ("Fastjet", "FN801", "13:30", 110),
    ],
}

def search_flights(origin: str, destination: str) -> str:
    """Searches available flights between two cities.

    Returns the airline, flight number, departure time and price in USD.
    """
    key = (origin.strip().lower(), destination.strip().lower())
    hits = FLIGHT_SCHEDULE.get(key, [])
    if not hits:
        return f"No flights found from {origin} to {destination}."
    return "\n".join(
        f"{airline} {number} departs {time} - ${price}"
        for airline, number, time, price in hits
    )

llm = OpenAI(model="gpt-5-nano")

agent = AgentWorkflow.from_tools_or_functions(
    [FunctionTool.from_defaults(search_flights)],
    llm=llm,
    system_prompt="You are a flight booking assistant. Always call the search tool before answering.",
)


In [9]:
response = await agent.run("Find me a flight from Nairobi to Mombasa.")
print(response)

Here are some Nairobi to Mombasa flight options I found:

- Jambojet JM872 — departs 17:05 from Nairobi, to Mombasa — $58
- Fly540 5F112 — departs 11:40 — $62
- Kenya Airways KQ203 — departs 07:15 — $85

Would you like me to book one of these? If yes, please tell me:
- Travel date
- Number of passengers
- Passenger full name (as on ID)
- Any seating or airline preferences

I can also look for more options or different times if you’d like.


### Adding Context

By default each `agent.run()` is stateless -- the agent has no memory of the previous call. Passing a `Context` object carries the conversation between runs.


In [10]:
# stateless 
await agent.run("I want to fly from Nairobi to Kigali.")

from llama_index.core.workflow import Context

ctx = Context(agent)

await agent.run("I want to fly from Nairobi to Kigali.", ctx=ctx)

response = await agent.run("What destination and origin did I give you?", ctx=ctx)
print(response)

Origin: Nairobi. Destination: Kigali.

For reference, I previously found these options Nairobi → Kigali: RwandAir WB404 (09:05, $140) and Kenya Airways KQ482 (16:20, $155).


### Creating RAG Agents with QueryEngineTools

A `QueryEngineTool` wraps a RAG query engine so an agent can call it like any other function. Here we index a set of airline policy documents, then give the agent a tool to look them up.

The `name` and `description` on the tool are what the model reads to decide when to call it, so they need to be specific.


In [11]:
from llama_index.core import VectorStoreIndex, Document
from llama_index.core.tools import QueryEngineTool
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding(model="text-embedding-3-small")

policy_documents = [
    Document(text="""Checked baggage: Economy passengers may check one bag up to 23 kg. Business class
passengers may check two bags up to 32 kg each. Carry-on is limited to one bag of 7 kg plus a
personal item such as a laptop bag or handbag."""),
    Document(text="""Refunds and changes: Tickets are fully refundable within 24 hours of booking with no
fee. After 24 hours, Economy tickets incur a $45 change fee plus any fare difference. Business
tickets are changeable free of charge. Tickets are non-refundable once the travel date has passed."""),
    Document(text="""Visa requirements: Passengers travelling on an East African passport may enter Kenya,
Rwanda, Tanzania and Uganda without a visa for up to 90 days. All other nationalities should confirm
entry requirements with the destination embassy before travel."""),
    Document(text="""Seat selection: Standard seat selection is free at check-in, which opens 48 hours
before departure. Advance seat selection including extra-legroom seats costs between $12 and $40
depending on the row."""),
    Document(text="""Loyalty programme: Members earn 1 point per US dollar spent. 25,000 points qualify for
Silver status, which includes priority boarding and an extra checked bag. 60,000 points qualify for
Gold status, which includes lounge access and free changes."""),
]

index = VectorStoreIndex.from_documents(policy_documents, embed_model=embed_model)

query_engine = index.as_query_engine(llm=llm)
tool = QueryEngineTool.from_defaults(
    query_engine,
    name="flight_policy_lookup",
    description="Looks up airline policies on checked baggage, refunds, changes, visas, seat selection and the loyalty programme. Use this tool for any policy question.",
)

rag_agent = AgentWorkflow.from_tools_or_functions(
    [tool],
    llm=llm,
    system_prompt="You are an airline support agent. Always call the policy lookup tool before answering.",
)

response = await rag_agent.run("What is the change fee for economy?")
print(response)

- After 24 hours from booking: $45 change fee, plus any fare difference.
- Within 24 hours of booking: economy tickets are fully refundable with no fee. 

If you have a specific itinerary in mind, I can check the fare difference for you.


### Creating Multi-agent systems

Most real agents need more than one capability. Instead of one agent with every tool, we can give each agent its own tools and let them route to each other.

Each agent gets a `name` and `description` -- the description is how the other agents decide when to delegate to it, so vague descriptions break routing.


In [ ]:
from llama_index.core.agent.workflow import FunctionAgent

flight_agent = FunctionAgent(
    name="flight_search",
    description="Searches available flights between two cities. Use for any flight search or booking request.",
    tools=[FunctionTool.from_defaults(search_flights)],
    llm=llm,
    system_prompt="You are a flight search agent. Always call the search tool before answering.",
)

policy_agent = FunctionAgent(
    name="policy_lookup",
    description="Answers questions about airline policies including baggage, refunds, visas and seat selection. Use for any policy question.",
    tools=[tool],
    llm=llm,
    system_prompt="You are a policy lookup agent. Always call the policy tool before answering.",
)

agent = AgentWorkflow(
    agents=[flight_agent, policy_agent],
    root_agent="flight_search",
)

response = await agent.run(
    "I want to fly Nairobi to Kigali. What flights are available, and how much baggage can I check?"
)
print(response)